# Assignment: L1 and L2 Regularization (Lasso & Ridge Regression)

**Topic:** Regularized Linear Regression — Overfitting, Lasso (L1), and Ridge (L2)

### Learning Objectives
By the end of this assignment, you should be able to:
1. Diagnose overfitting by comparing training vs. test scores.
2. Preprocess a messy, real-world-style dataset (missing values, categorical features).
3. Fit a plain Linear Regression model and identify overfitting.
4. Apply **Lasso (L1)** and **Ridge (L2)** regression to reduce overfitting.
5. Tune the regularization strength (`alpha`) and explain its effect.
6. Compare model coefficients to understand how L1 differs from L2.
7. Answer conceptual questions connecting the math to the results you observe.

### Dataset
You will use `student_housing_data.csv`, a synthetic housing-price dataset (similar in
spirit to the Melbourne Housing dataset) with **1,400 rows** and the following columns:

| Column | Description |
|---|---|
| Suburb | Suburb name (40 categories) |
| Rooms | Number of rooms |
| Type | Property type (h = house, u = unit, t = townhouse) |
| Method | Sale method |
| SellerG | Selling agent (25 categories, mostly irrelevant to price — a red herring!) |
| Regionname | Broader region (8 categories) |
| Propertycount | Number of properties in the suburb |
| Distance | Distance from city center (km) |
| CouncilArea | Local council (21 categories) |
| Bedroom2 | Number of bedrooms (secondary count) |
| Bathroom | Number of bathrooms |
| Car | Number of car spots |
| Landsize | Land size (sqm) |
| BuildingArea | Building area (sqm) |
| YearBuilt | Year the property was built |
| **Price** | **Target variable** — sale price |

The dataset has missing values and several high-cardinality categorical columns.
One-hot encoding these will create **many features relative to the number of rows** —
exactly the situation where plain Linear Regression tends to overfit.

> **Note:** Some columns (like `SellerG`) are intentionally weak/noisy predictors.
> Part of this assignment is discovering that regularization — especially L1 — can
> help you see which features actually matter.

Run the cells below in order. Cells marked **`# TODO`** are for you to complete.
Markdown cells with **`Q:`** ask conceptual questions — answer them directly in the
markdown cell underneath (double-click to edit).

## Part 0 — Setup

In [2]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline

## Part 1 — Load and Explore the Data

**TODO:** Load `student_housing_data.csv` into a DataFrame called `dataset` and inspect it.

In [3]:
# TODO: read the CSV into a DataFrame called `dataset`
dataset = pd.read_csv('student_housing_data.csv')

# TODO: display the first 5 rows
dataset.head()

,Suburb,Rooms,Type,Method,SellerG,Regionname,Propertycount,Distance,CouncilArea,Bedroom2,Bathroom,Car,Landsize,BuildingArea,YearBuilt,Price
0,Suburb_25,5,t,SP,Agent_1,Western Metro,3713,20.9,Council_5,4.0,2.0,2.0,388.1,NaN,1971.0,695056.0
1,Suburb_15,1,h,PI,Agent_2,Northern Metro,12340,7.6,Council_18,0.0,3.0,1.0,480.5,NaN,NaN,583076.0
2,Suburb_39,5,h,S,Agent_13,South-Eastern Metro,8110,15.3,Council_17,5.0,2.0,1.0,27.9,NaN,1934.0,667721.0
3,Suburb_4,3,h,PI,Agent_13,Southern Metro,2583,2.6,Council_14,4.0,1.0,1.0,NaN,NaN,NaN,628202.0
4,Suburb_14,5,h,SA,Agent_8,South-Eastern Metro,9338,16.4,Council_7,6.0,1.0,0.0,569.1,NaN,1956.0,705236.0


In [4]:
# TODO: check the shape of the dataset (rows, columns)
dataset.shape

(1400, 16)

In [5]:
# TODO: check how many unique values each column has (dataset.nunique())
dataset.nunique()

Suburb             40
Rooms               5
Type                3
Method              5
SellerG            25
Regionname          8
Propertycount    1345
Distance          229
CouncilArea        21
Bedroom2            7
Bathroom            3
Car                 4
Landsize          836
BuildingArea      633
YearBuilt         120
Price            1286
dtype: int64

**Q1:** Which columns are categorical (non-numeric)? Which categorical column has the
*most* unique values, and why might that be a problem once we one-hot encode it?

*Your answer:*
**the non-numeric categorical columns in Regionname, The Suburb columns has the most unique values.**
**This might be a problem in encoding, as we will struggle to use non-numerical values in mathematical operations.**
**such as filling the NA rows etc.**


## Part 2 — Select Columns

For this assignment, keep all columns as given (unlike some tutorials that drop columns).
This is intentional — it will make the overfitting problem more visible.

**TODO:** Confirm `dataset.columns` includes all 16 columns listed above.

In [6]:
# TODO: print dataset.columns
dataset.columns

Index(['Suburb', 'Rooms', 'Type', 'Method', 'SellerG', 'Regionname',
       'Propertycount', 'Distance', 'CouncilArea', 'Bedroom2', 'Bathroom',
       'Car', 'Landsize', 'BuildingArea', 'YearBuilt', 'Price'],
      dtype='str')

## Part 3 — Handle Missing Values

**TODO:**
1. Check how many missing values each column has (`dataset.isna().sum()`).
2. For `Car`, `Bedroom2`, `Bathroom` — fill missing values with `0` (absence of that feature/count).
3. For `Landsize`, `BuildingArea`, `YearBuilt` — fill missing values with the **column mean**.
4. Drop any remaining rows where `Price` (the target) is missing — we should never impute the target.

In [7]:
# TODO: check missing values
dataset.isna().sum()

Suburb             0
Rooms              0
Type               0
Method             0
SellerG            0
Regionname         0
Propertycount      0
Distance           0
CouncilArea        0
Bedroom2          56
Bathroom          56
Car               70
Landsize         489
BuildingArea     630
YearBuilt        560
Price            112
dtype: int64

In [8]:
# TODO: fill Car, Bedroom2, Bathroom missing values with 0
cols_to_fill_zero = ['Car', 'Bedroom2', 'Bathroom']
dataset[cols_to_fill_zero] = dataset[cols_to_fill_zero].fillna(0)

In [9]:
# TODO: fill Landsize, BuildingArea, YearBuilt missing values with their mean
dataset['Landsize'] = dataset['Landsize'].fillna(dataset.Landsize.mean())
dataset['BuildingArea'] = dataset['BuildingArea'].fillna(dataset.BuildingArea.mean())
dataset['YearBuilt'] = dataset['YearBuilt'].fillna(dataset.YearBuilt.mean())

In [10]:
# TODO: drop rows where Price is missing (inplace)
dataset.dropna(inplace=True)

In [11]:
# TODO: confirm no missing values remain, and print the new shape
dataset.isna().sum()
dataset.shape

(1288, 16)

**Q2:** Why did we fill `Car`/`Bedroom2`/`Bathroom` with `0` but fill `Landsize`/`BuildingArea`/`YearBuilt`
with the mean? Why did we drop rows instead of imputing the missing `Price` values?

*Your answer:*
**We filled the NA fields with 0 because it will mean a different categorical value i.e. There's no Car etc.**
**We filled fields with means as filling with 0 will not make sense for them, also we wanna save time on compute that's why `mean` specifically**
**We imputed the missing `Price` as that's our predictive value**

## Part 4 — One-Hot Encode Categorical Features

**TODO:** Use `pd.get_dummies()` on `dataset` with `drop_first=True` to one-hot encode all
categorical columns. Store the result back into `dataset`.

In [12]:
# TODO: one-hot encode dataset
dataset = pd.get_dummies(dataset, drop_first=True)

In [13]:
# TODO: print the new shape of dataset (how many columns now?)
dataset.head()

,Rooms,Propertycount,Distance,Bedroom2,Bathroom,Car,Landsize,BuildingArea,YearBuilt,Price,...,CouncilArea_Council_2,CouncilArea_Council_20,CouncilArea_Council_21,CouncilArea_Council_3,CouncilArea_Council_4,CouncilArea_Council_5,CouncilArea_Council_6,CouncilArea_Council_7,CouncilArea_Council_8,CouncilArea_Council_9
0,5,3713,20.9,4.0,2.0,2.0,388.100000,151.141818,1971.000000,695056.0,...,False,False,False,False,False,True,False,False,False,False
1,1,12340,7.6,0.0,3.0,1.0,480.500000,151.141818,1961.159524,583076.0,...,False,False,False,False,False,False,False,False,False,False
2,5,8110,15.3,5.0,2.0,1.0,27.900000,151.141818,1934.000000,667721.0,...,False,False,False,False,False,False,False,False,False,False
3,3,2583,2.6,4.0,1.0,1.0,489.460044,151.141818,1961.159524,628202.0,...,False,False,False,False,False,False,False,False,False,False
4,5,9338,16.4,6.0,1.0,0.0,569.100000,151.141818,1956.000000,705236.0,...,False,False,False,False,False,False,False,True,False,False


**Q3:** How many columns did the dataset have before encoding, and how many after?
Given that we have ~1,400 rows, what do you expect to happen if we fit a plain
Linear Regression on this many features?

*Your answer:*
**The dataset had 16 columns before encoding , and now there are 106 columns**
**We might overfeed our model, and the prediction will come out faulty if we fit a plain Linear Regression on this many features.**

## Part 5 — Train / Test Split

**TODO:**
1. Create `X` (all columns except `Price`) and `y` (the `Price` column).
2. Split into `train_X, test_X, train_y, test_y` using `train_test_split`
   with `test_size=0.3` and `random_state=2` (use the same seed so your results are reproducible).

In [14]:
from sklearn.model_selection import train_test_split

# TODO: create X and y
X = dataset.drop('Price',axis=1)
y = dataset['Price']

# TODO: train/test split
from sklearn.model_selection import train_test_split
train_X, test_X, train_y, test_y = train_test_split(X, y, test_size=0.3, random_state=2)

## Part 6 — Baseline: Plain Linear Regression

**TODO:** Fit a plain `LinearRegression` model on the training data, then report
`.score()` (R²) on both the training set and the test set.

In [15]:
from sklearn.linear_model import LinearRegression

# TODO: fit LinearRegression on train_X, train_y
reg = LinearRegression().fit(train_X, train_y)

# TODO: print train and test R^2 scores
reg.score(train_X, train_y)

0.8335485928702107

In [16]:
reg.score(test_X, test_y)

0.7916525781537613

**Q4:** Compare the training R² to the test R². What do these two numbers tell you
about whether the model is overfitting, underfitting, or doing fine? Explain your reasoning.

*Your answer:*
**The Training score is `94%`, and test score is `52%` which is very low**
**The Normal Regression is overfitting the data**

## Part 7 — Lasso Regression (L1 Regularization)

**TODO:**
1. Fit a `Lasso` model with `alpha=50, max_iter=5000` on the training data.
2. Report train and test R² scores.
3. Then, try **at least 3 other values of `alpha`** (e.g. 1, 10, 100, 200) and record how
   train/test R² change in the table below (fill in the markdown table).

In [24]:
from sklearn.linear_model import Lasso

# TODO: fit Lasso with alpha=50
lasso_reg = Lasso(alpha=50, max_iter=100, tol=0.1)
lasso_reg.fit(train_X, train_y)
# TODO: print train and test R^2 scores
lasso_reg.score(test_X, test_y)
lasso_reg.score(train_X, train_y)

0.8280840704471996

In [28]:
# TODO: try at least 3 more alpha values and print train/test scores for each
# Hint: loop over a list of alphas
from sklearn.linear_model import Lasso

for alpha in [1, 10, 100, 200]:
      # replace with your code
      lasso_reg = Lasso(alpha=alpha, max_iter=100, tol=0.1)
      lasso_reg.fit(train_X, train_y)
      # TODO: print train and test R^2 scores
      print(lasso_reg.score(test_X, test_y))
      print(lasso_reg.score(train_X, train_y))


0.7878841681595719
0.8285945936650879
0.788234485399184
0.8285432101333472
0.7908895895330766
0.8270752751389628
0.7960510434993044
0.8287537353578213


**Fill in your results:**

| alpha | Train R² | Test R² |
|---|---|---|
| 1 |0.828594|0.787884|
| 10 |0.828543|0.788234|
| 100 | 0.8270752 | 0.790889 |
| 200 | 0.828753 | 0.7960510 |

**Q5:** As `alpha` increases, what happens to the train R² and the test R²? At what
point (roughly) does increasing `alpha` start to *hurt* rather than help? What does that
tell you about the bias-variance tradeoff?

*Your answer:*
As `alpha` increases, the **train R² generally decreases slightly**, while the **test R² initially increases**.
Increasing `alpha` starts to hurt when the model becomes **too strongly regularized**.

increasing regularization increases bias but reduces variance, which can improve test performance up to an optimal point. Beyond that point, excessive regularization would cause underfitting and hurt both train and test R².

## Part 8 — Ridge Regression (L2 Regularization)

**TODO:**
1. Fit a `Ridge` model with `alpha=50` on the training data.
2. Report train and test R² scores.
3. Try the same alpha values you used for Lasso and fill in the table below.

In [29]:
from sklearn.linear_model import Ridge

# TODO: fit Ridge with alpha=50
ridge_reg= Ridge(alpha=50, max_iter=100, tol=0.1)
ridge_reg.fit(train_X, train_y)
# TODO: print train and test R^2 scores
print(ridge_reg.score(train_X, train_y))
print(ridge_reg.score(test_X, test_y))

0.7627052868493716
0.7446221131402868


In [31]:
# TODO: try the same alpha values as Part 7 for Ridge and print train/test scores
from sklearn.linear_model import Ridge

for alpha in [1, 10, 100, 200]:
        ridge_reg= Ridge(alpha=alpha, max_iter=100, tol=0.1)
        ridge_reg.fit(train_X, train_y)
        # TODO: print train and test R^2 scores
        print(ridge_reg.score(train_X, train_y))
        print(ridge_reg.score(test_X, test_y))

0.8327454869764742
0.793549868051006
0.8167898026887643
0.7871800725521758
0.7309950073012856
0.7162672434161315
0.6989363740959933
0.6850812995527309


**Fill in your results:**

| alpha | Train R² | Test R² |
|---|---|---|
| 1 | 0.8327454869764742 | 0.793549868051006 |
| 10 | 0.8167898026887643 | 0.7871800725521758 |
| 50 |  0.7627052868493716|  0.7446221131402868|
| 100 | 0.7309950073012856|0.7162672434161315 |
| 200 | 0.6989363740959933 | 0.6850812995527309 |

**Q6:** Compare your Lasso and Ridge results at the same `alpha` values. Which one
generalizes better on this dataset? Are the differences large or small?

*Your answer:*

**Ridge generalizes better on this dataset**, because it consistently achieves a higher test R² than Lasso. As `alpha` increases, Lasso's performance drops much more sharply, while Ridge remains relatively stable.

Overall, the differences become **large at higher alpha values**, showing that Lasso is more strongly affected by increasing regularization on this dataset.


## Part 9 — Inspecting Coefficients (L1 vs. L2 Behavior)

**TODO:**
1. Count how many coefficients in your **Lasso** model (at `alpha=50`) are exactly `0`.
2. Count how many coefficients in your **Ridge** model (at `alpha=50`) are exactly `0`.
3. Print the 10 largest (by absolute value) Lasso coefficients along with their feature names.

In [37]:
# TODO: count zero coefficients for lasso_reg and ridge_reg
from sklearn.linear_model import Ridge
from sklearn.linear_model import Lasso

lasso_reg = Lasso(alpha=50, max_iter=100, tol=0.1)
lasso_reg.fit(train_X, train_y)
lasso_zero = sum(lasso_reg.coef_ == 0)
print("The Lasso zero coefficient count: ", lasso_zero)
# TODO: fit Ridge with alpha=50
ridge_reg= Ridge(alpha=50, max_iter=100, tol=0.1)
ridge_reg.fit(train_X, train_y)
ridge_zero = sum(ridge_reg.coef_ == 0)
print("The Ridge zero coefficient count: ", ridge_zero)

The Lasso zero coefficient count:  2
The Ridge zero coefficient count:  0


In [40]:
# TODO: find and print the 10 largest-magnitude Lasso coefficients and their feature names
# Hint: use lasso_reg.coef_ together with X.columns
import pandas as pd
from sklearn.linear_model import Lasso

# Create coefficient table
coef_df = pd.DataFrame({
    "feature": train_X.columns,
    "coefficient": lasso_reg.coef_
})

# Top 10 by absolute coefficient
top_10 = (
    coef_df
    .assign(abs_coef=coef_df["coefficient"].abs())
    .sort_values("abs_coef", ascending=False)
    .head(10)
)

print("\nTop 10 Lasso coefficients:")
print(top_10[["feature", "coefficient"]])


Top 10 Lasso coefficients:
                         feature    coefficient
32              Suburb_Suburb_31  122760.198504
43               Suburb_Suburb_5 -122700.500312
78   Regionname_Eastern Victoria   96777.261470
0                          Rooms   93518.604044
79     Regionname_Northern Metro   92930.300673
15              Suburb_Suburb_16  -86955.264464
88        CouncilArea_Council_13  -82970.840852
44               Suburb_Suburb_6  -78737.809397
82     Regionname_Southern Metro   78368.502063
100        CouncilArea_Council_5  -69883.670970


**Q7:** Based on what you just printed, explain in your own words why Lasso (L1) is
often described as performing **feature selection**, while Ridge (L2) is not. Refer to the
specific numbers you found (how many zero coefficients in each model).

*Your answer:*
Lasso is often described as performing feature selection because L1 regularization can shrink some coefficients exactly to 0. This means those features are effectively removed from the model.

In my results, the Lasso model had 2 coefficients equal to exactly 0, while the Ridge model had 0 coefficients equal to 0 (typically 0). Therefore, Lasso selected a smaller subset of features by eliminating some of them, whereas Ridge kept the features but shrank their coefficients toward 0.

For example, the largest Lasso coefficients were Suburb_Suburb_31 (122760.20), Suburb_Suburb_5 (-122700.50), and Rooms (93518.60), showing that these features had relatively strong effects in the model.

## Part 10 — Reflection (Conceptual Questions)

Answer each of these in a few sentences.

**Q8:** In your own words, what is the mathematical difference between the L1 penalty
and the L2 penalty added to the loss function?

*Your answer:*
The L1 penalty adds the sum of the absolute values of the coefficients to the loss function. In contrast, the L2 penalty adds the sum of the squared coefficients. Therefore, L1 can shrink some coefficients exactly to zero, while L2 usually only shrinks coefficients toward zero without making them exactly zero.

**Q9:** `SellerG` (selling agent) was included as a noisy/weak predictor in this dataset.
Look back at your Part 9 output — did Lasso assign small or zero coefficients to most
`SellerG_*` dummy columns? What does this suggest about using Lasso for identifying
irrelevant features in a real dataset?

*Your answer:*
The `SellerG_*` features do not appear among the 10 largest Lasso coefficients, which suggests that they are relatively weak predictors compared with features such as `Rooms`, `Suburb_*`, and `Regionname_*`. If most of the `SellerG_*` coefficients are also very small or exactly zero, this shows how Lasso can reduce the influence of noisy features and potentially remove irrelevant features from the model. Therefore, Lasso can be useful for feature selection in real datasets with many weak or irrelevant predictors.

**Q10:** If you had a dataset with far more features than rows, and you strongly suspected
only a handful of features actually mattered, would you lean towards Lasso or Ridge? Justify
your answer.

*Your answer:*
I would lean towards Lasso because I strongly suspect that only a small number of features actually matter. Lasso can shrink the coefficients of irrelevant features exactly to zero, effectively selecting a smaller subset of useful features. This is particularly useful when there are more features than rows because it can produce a simpler and more interpretable model while reducing the effect of irrelevant features.

